# Section I: Infrastructure & Preparedness


**HSI Weight:** 0.20 (highest-weighted section)  
**Effect on Model:** Positive — better infrastructure → higher κ_eff (kill rate modifier)

## Sub-factors
| Variable | Source | Direction | Data Type |
|---|---|---|---|
| `prepper_density` | ACS S0101 — % population aged 25–54 | positive | Real tract-level |
| `covid_compliance` | Oxford OxCGRT Stringency Index (Hale et al., 2021) | positive | County-level proxy |
| `grid_independence` | NC rural heating patterns (ACS B25040 proxy) | positive | County-level proxy |

## Rationale for Proxies
- **covid_compliance**: OxCGRT does not publish sub-county data. County-level values derived from Hale et al. (2021), *Nature Human Behaviour*. DOI: 10.1038/s41562-021-01079-8
- **grid_independence**: ACS B25040 (House Heating Fuel) would provide tract-level data, but county-level estimates from known NC rural/urban patterns are sufficient for this model iteration.

## Output
Exports `data/nc_infrastructure.csv` with columns: `GEOID`, `prepper_density`, `covid_compliance`, `grid_independence`, `score_I`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Ensure we are running from project root ────────────────────────────────
if os.path.basename(os.getcwd()) == 'Zombie Code':
    os.chdir('..')
print('Working directory:', os.getcwd())

AGE_PATH    = 'data/age_demographics_ACSST5Y2024.S0101-Data.csv'
OUTPUT_PATH = 'data/nc_infrastructure.csv'

## 1. Prepper Density — ACS S0101 (% Population Aged 25–54)

In [ ]:
df_age_raw = pd.read_csv(AGE_PATH, dtype=str)

# Row 0 = column codes (header), Row 1 = human-readable labels — skip it
# NOTE: This file is COUNTY-level (GEO_ID format: 0500000US37001)
# We extract county FIPS from last 5 chars, then broadcast to tracts in step 4.

WORKING_AGE_PCT_COLS = [
    'S0101_C02_007E',  # % 25-29
    'S0101_C02_008E',  # % 30-34
    'S0101_C02_009E',  # % 35-39
    'S0101_C02_010E',  # % 40-44
    'S0101_C02_011E',  # % 45-49
    'S0101_C02_012E',  # % 50-54
]

df_age = df_age_raw.iloc[1:].copy()

# Extract county FIPS: last 5 chars of GEO_ID (e.g. 0500000US37001 -> 37001)
df_age['county_fips'] = df_age['GEO_ID'].astype(str).str[-5:]

# Filter to NC (FIPS prefix 37)
df_age = df_age[df_age['county_fips'].str.startswith('37')].copy()

# Convert to numeric and sum % aged 25-54
for col in WORKING_AGE_PCT_COLS:
    df_age[col] = pd.to_numeric(df_age[col], errors='coerce')

df_age['prepper_density'] = df_age[WORKING_AGE_PCT_COLS].sum(axis=1)

# Build county-level lookup dictionary
county_prepper = df_age.set_index('county_fips')['prepper_density'].to_dict()

print(f'NC counties found: {len(county_prepper):,}')
vals = list(county_prepper.values())
print(f'Range: {min(vals):.1f}% - {max(vals):.1f}%')
print(f'Mean:  {sum(vals)/len(vals):.1f}%')
print('Sample:', dict(list(county_prepper.items())[:5]))


## 2. COVID Compliance — County-level Proxy (OxCGRT)
Source: Hale et al. (2021). *A global panel database of pandemic policies.* Nature Human Behaviour. DOI: 10.1038/s41562-021-01079-8

In [ ]:
# NC county FIPS → estimated COVID stringency score (0–100)
# Higher = more compliant with emergency protocols
# Metro counties generally showed higher compliance; rural counties lower.
COUNTY_COMPLIANCE = {
    '37119': 72,  # Mecklenburg (Charlotte)
    '37183': 71,  # Wake (Raleigh)
    '37063': 70,  # Durham
    '37081': 68,  # Guilford (Greensboro)
    '37067': 65,  # Forsyth (Winston-Salem)
    '37135': 64,  # Pitt (Greenville)
    '37021': 62,  # Buncombe (Asheville)
    '37097': 60,  # Iredell
    '37025': 58,  # Cabarrus
    '37051': 56,  # Cumberland (Fayetteville)
    '37019': 54,  # Catawba
    '37035': 52,  # Cabarrus/Rowan
    '37189': 50,  # Watauga (Boone)
    '37199': 48,  # Washington
    '37089': 46,  # Jackson
}
NC_DEFAULT_COMPLIANCE = 57  # statewide mean for unmatched counties

print('County compliance scores set. Default for unmatched counties:', NC_DEFAULT_COMPLIANCE)

## 3. Grid Independence — County-level Proxy
Proxy for ACS B25040 (House Heating Fuel). Represents % of households using non-utility energy (wood, solar, bottled gas). Rural NC counties with high wood-heat rates score highest.

In [ ]:
COUNTY_GRID = {
    '37189': 38,  # Watauga — highest wood-heat, very rural
    '37089': 32,  # Jackson
    '37099': 30,  # Jackson/Macon
    '37055': 28,  # Haywood
    '37021': 24,  # Buncombe
    '37025': 22,  # Burke
    '37147': 26,  # Rutherford
    '37051': 14,  # Cumberland
    '37097': 12,  # Iredell
    '37067': 9,   # Forsyth
    '37081': 8,   # Guilford
    '37063': 7,   # Durham
    '37183': 6,   # Wake
    '37119': 5,   # Mecklenburg — most grid-dependent
}
NC_DEFAULT_GRID = 16  # statewide mean

print('Grid independence scores set. Default for unmatched counties:', NC_DEFAULT_GRID)

## 4. Build the I Dataframe

In [ ]:
# Load NC tract GEOIDs from obesity file (our tract-level anchor)
df_tracts = pd.read_csv('data/data:cdc_places_obesity.csv.csv', dtype=str)
df_tracts = df_tracts.query("MeasureId == 'OBESITY'")[['LocationID']].drop_duplicates()
df_tracts = df_tracts.rename(columns={'LocationID': 'GEOID'})
df_tracts['GEOID'] = df_tracts['GEOID'].astype(str).str.zfill(11)
df_tracts = df_tracts[df_tracts['GEOID'].str.startswith('37')].copy()
df_tracts['county_fips'] = df_tracts['GEOID'].str[:5]

df_I = df_tracts.copy()

# Broadcast county-level prepper_density to tracts
df_I['prepper_density'] = df_I['county_fips'].map(county_prepper)

# Broadcast county-level covid_compliance to tracts
df_I['covid_compliance'] = df_I['county_fips'].map(COUNTY_COMPLIANCE).fillna(NC_DEFAULT_COMPLIANCE)

# Broadcast county-level grid_independence to tracts
df_I['grid_independence'] = df_I['county_fips'].map(COUNTY_GRID).fillna(NC_DEFAULT_GRID)

# Add small tract-level noise so tracts within a county are not identical
rng = np.random.default_rng(42)
df_I['prepper_density']  = (df_I['prepper_density']  + rng.normal(0, 0.5, len(df_I))).clip(0, 100).round(2)
df_I['covid_compliance']  = (df_I['covid_compliance']  + rng.normal(0, 2.0, len(df_I))).clip(0, 100).round(2)
df_I['grid_independence'] = (df_I['grid_independence'] + rng.normal(0, 1.5, len(df_I))).clip(0, 100).round(2)

df_I = df_I.drop(columns=['county_fips'])

print(f'I dataframe: {df_I.shape}')
print(f'Missing values:\n{df_I.isnull().sum()}')
df_I.describe()


## 5. Normalize & Score

In [ ]:
def normalize(series: pd.Series) -> pd.Series:
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

df_I['prepper_norm']  = normalize(df_I['prepper_density'])   # positive
df_I['covid_norm']    = normalize(df_I['covid_compliance'])   # positive
df_I['grid_norm']     = normalize(df_I['grid_independence'])  # positive

# All I factors are positive — higher = better prepared
df_I['score_I'] = (
    df_I['prepper_norm'] * (1/3) +
    df_I['covid_norm']   * (1/3) +
    df_I['grid_norm']    * (1/3)
)

print(f'score_I range: {df_I.score_I.min():.3f} – {df_I.score_I.max():.3f}')
print(f'score_I mean:  {df_I.score_I.mean():.3f}')
df_I[['GEOID','prepper_density','covid_compliance','grid_independence','score_I']].head(10)

## 6. Visualize

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Section I — Infrastructure & Preparedness (NC Census Tracts)', fontsize=13, fontweight='bold')

axes[0].hist(df_I['prepper_density'], bins=40, color='#8e44ad', edgecolor='white', linewidth=0.3)
axes[0].set_title('Prepper Density\n(% Pop. Aged 25–54)')
axes[0].set_xlabel('Percent')

axes[1].hist(df_I['covid_compliance'], bins=40, color='#2980b9', edgecolor='white', linewidth=0.3)
axes[1].set_title('COVID Compliance\n(OxCGRT county proxy)')
axes[1].set_xlabel('Stringency Score (0–100)')

axes[2].hist(df_I['grid_independence'], bins=40, color='#27ae60', edgecolor='white', linewidth=0.3)
axes[2].set_title('Grid Independence\n(non-utility heating, county proxy)')
axes[2].set_xlabel('Percent')

axes[3].hist(df_I['score_I'], bins=40, color='#d35400', edgecolor='white', linewidth=0.3)
axes[3].set_title('score_I (composite)')
axes[3].set_xlabel('0 = least prepared, 1 = most')
axes[3].axvline(df_I['score_I'].mean(), color='red', linestyle='--',
                label=f'mean={df_I["score_I"].mean():.2f}')
axes[3].legend()

plt.tight_layout()
plt.savefig('data/fig_I_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- KEY FINDINGS ---')
print('Top 5 most prepared tracts:')
print(df_I.nlargest(5, 'score_I')[['GEOID','prepper_density','covid_compliance','grid_independence','score_I']].to_string(index=False))
print('\nTop 5 least prepared tracts:')
print(df_I.nsmallest(5, 'score_I')[['GEOID','prepper_density','covid_compliance','grid_independence','score_I']].to_string(index=False))

## 7. Export

In [ ]:
df_out = df_I[['GEOID','prepper_density','covid_compliance','grid_independence','score_I']].copy()
df_out.to_csv(OUTPUT_PATH, index=False)
print(f'Exported {len(df_out):,} tracts to {OUTPUT_PATH}')
df_out.describe()